# Countdown GRPO: training and evaluation

Runs the whole pipeline on Colab (a T4 is enough): training (60 then 150 steps), paired evaluation, null model.
Code structure is described in `README.md`, results in `countdown_grpo_report.pdf`.

Run the cells in order. Training is resumable: checkpoints are written to Drive every 20 steps.

## 0. Setup

In [ ]:
!pip -q install trl transformers datasets accelerate peft

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Get the project files into the working directory. Either clone the repository
# (set REPO_URL) or upload the five .py files with the Colab file browser.
REPO_URL = ""   # for example "https://github.com/leopmr/countdown-grpo"
if REPO_URL:
    !git clone {REPO_URL} /content/countdown-grpo
    %cd /content/countdown-grpo
!ls *.py

In [ ]:
# Record library versions (keep them next to the results)
!pip freeze | grep -iE "^(trl|peft|transformers|torch|accelerate|datasets)=="
!nvidia-smi --query-gpu=name,compute_cap --format=csv

In [ ]:
# T4 only (compute capability 7.5): with TRL 1.15.0 the bf16 to fp32 matmul path of the
# chunked log-prob kernel is unsupported. Fall back to float32. Skip on newer GPUs.
import torch
if torch.cuda.get_device_capability() < (8, 0):
    import trl.kernels.chunked_logprob as k
    k._MM_OUT_DTYPE = False
    print("T4 workaround applied")

In [ ]:
OUT = "/content/drive/MyDrive/countdown-grpo-v3"      # checkpoints and adapter
RES = "/content/drive/MyDrive/countdown-eval"         # evaluation JSON files

## 1. Sanity check of the rewards (no GPU needed)

In [ ]:
!python countdown_grpo_starter.py

## 2. Training

Phase 1: 60 steps (4 puzzles per step, group size 8).

In [ ]:
from trl import GRPOTrainer
from train_countdown_grpo import build_hf_dataset, make_config, REWARD_FUNCS, peft_config, MODEL_NAME

config = make_config(output_dir=OUT, max_steps=60)
trainer = GRPOTrainer(model=MODEL_NAME, reward_funcs=REWARD_FUNCS, args=config,
                      train_dataset=build_hf_dataset(), peft_config=peft_config)
trainer.train()
trainer.save_model(OUT)

Phase 2: continue to 150 steps. `max_steps` comes from the new config, and the Trainer restores step and optimizer from the checkpoint, skipping the batches already consumed. Restart the runtime first if GPU memory is still held by phase 1 (then re-run the setup cells, including the T4 workaround).

In [ ]:
from trl import GRPOTrainer
from train_countdown_grpo import build_hf_dataset, make_config, REWARD_FUNCS, peft_config, MODEL_NAME

config = make_config(output_dir=OUT, max_steps=150)
trainer = GRPOTrainer(model=MODEL_NAME, reward_funcs=REWARD_FUNCS, args=config,
                      train_dataset=build_hf_dataset(), peft_config=peft_config)
trainer.train(resume_from_checkpoint=f"{OUT}/checkpoint-60")
trainer.save_model(OUT)

## 3. Evaluation

Each run writes a JSON file to `RES`. The baseline is evaluated without `--adapter`. Greedy on 300 puzzles first.

In [ ]:
!python eval_countdown.py --mode mid --tag baseline-mid --out-dir {RES}
!python eval_countdown.py --mode mid --adapter {OUT}/checkpoint-150 --tag step150-mid --out-dir {RES}

Sampling at T=1.0 (the training temperature), 200 puzzles x 4 completions.

In [ ]:
!python eval_countdown.py --mode samp --temperature 1.0 --seed 0 --skip-greedy --tag baseline-T1 --out-dir {RES}
!python eval_countdown.py --mode samp --temperature 1.0 --seed 0 --skip-greedy --adapter {OUT}/checkpoint-150 --tag step150-T1 --out-dir {RES}

Hard puzzles (4 numbers), greedy and sampled.

In [ ]:
!python eval_countdown.py --mode hard --temperature 1.0 --seed 0 --tag baseline-hard --out-dir {RES}
!python eval_countdown.py --mode hard --temperature 1.0 --seed 0 --adapter {OUT}/checkpoint-150 --tag step150-hard --out-dir {RES}

## 4. Paired comparisons (baseline first)

In [ ]:
import glob
def latest(mode, tag):
    files = sorted(glob.glob(f"{RES}/*_{mode}_{tag}.json"))
    assert files, f"no file for {mode} {tag}"
    return files[-1]

pairs = [("mid", "baseline-mid", "step150-mid", ["greedy"]),
         ("samp", "baseline-T1", "step150-T1", ["sampled"]),
         ("hard", "baseline-hard", "step150-hard", ["hard", "hard_sampled"])]
for mode, a, b, sections in pairs:
    for sec in sections:
        A, B = latest(mode, a), latest(mode, b)
        print("=" * 70, f"\n{mode} / {sec}\n")
        !python compare_evals.py {A} {B} --section {sec}

## 5. Null model (exact accuracy of a random valid expression)

In [ ]:
!python null_model.py